In [ ]:
import pandas as pd
import numpy as np

train_ratings = pd.read_csv(
    "../datasets/processed/train_ratings.csv"
)

test_ratings = pd.read_csv(
    "../datasets/processed/test_ratings.csv"
)

recipes = pd.read_csv(
    "../../datasets/RAW_recipes.csv"
)

print("Train shape:", train_ratings.shape)
print("Test shape:", test_ratings.shape)
print("Recipes shape:", recipes.shape)

In [ ]:
num_users = train_ratings["user_id"].nunique()
num_recipes = train_ratings["recipe_id"].nunique()

print("Users in train:", num_users)
print("Recipes in train:", num_recipes)
print("Ratings in train:", len(train_ratings))

In [ ]:
user_ids = train_ratings["user_id"].unique()
recipe_ids = train_ratings["recipe_id"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

recipe_to_index = {
    recipe_id: index
    for index, recipe_id in enumerate(recipe_ids)
}

print("Number of users:", len(user_to_index))
print("Number of recipes:", len(recipe_to_index))

In [ ]:
from scipy.sparse import csr_matrix

rows = train_ratings["user_id"].map(user_to_index)
cols = train_ratings["recipe_id"].map(recipe_to_index)
values = train_ratings["rating"]

user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(len(user_to_index), len(recipe_to_index))
)

print("User-item matrix shape:", user_item_matrix.shape)
print("Number of stored ratings:", user_item_matrix.nnz)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

test_user_id = test_ratings["user_id"].iloc[0]

user_index = user_to_index[test_user_id]

user_vector = user_item_matrix[user_index]

similarities = cosine_similarity(
    user_vector,
    user_item_matrix
).flatten()

similarities[user_index] = -1

similar_users_indices = np.argsort(similarities)[::-1][:10]

print("Test user:", test_user_id)
print("Most similar users:")
print(similar_users_indices)

In [ ]:
similar_users = pd.DataFrame({
    "user_id": [user_ids[i] for i in similar_users_indices],
    "similarity": [similarities[i] for i in similar_users_indices]
})

similar_users

In [ ]:
# Recepti koje je test korisnik već ocenio
user_seen_recipes = set(
    train_ratings[
        train_ratings["user_id"] == test_user_id
        ]["recipe_id"]
)

# Uzmi interakcije sličnih korisnika
similar_user_ids = similar_users["user_id"].tolist()

candidate_ratings = train_ratings[
    train_ratings["user_id"].isin(similar_user_ids)
].copy()

# Izbaci recepte koje je naš korisnik već video
candidate_ratings = candidate_ratings[
    ~candidate_ratings["recipe_id"].isin(user_seen_recipes)
].copy()

print("Recipes already seen by test user:", len(user_seen_recipes))
print("Candidate interactions:", len(candidate_ratings))
print("Candidate recipes:", candidate_ratings["recipe_id"].nunique())

In [ ]:
# Dodaj similarity svakom user-u
candidate_ratings = candidate_ratings.merge(
    similar_users,
    on="user_id",
    how="left"
)

# Weighted score:
# ocena * similarity korisnika
candidate_ratings["weighted_rating"] = (
        candidate_ratings["rating"] * candidate_ratings["similarity"]
)

# Zbirni score za svaki recept
recipe_scores = (
    candidate_ratings
    .groupby("recipe_id")
    .agg(
        score=("weighted_rating", "sum"),
        supporting_users=("user_id", "nunique")
    )
    .reset_index()
    .sort_values("score", ascending=False)
)

print("Number of candidate recipes:", len(recipe_scores))

recipe_scores.head(10)

In [ ]:
top_10_recommendations = (
    recipe_scores
    .head(10)
    .merge(
        recipes[["id", "name"]],
        left_on="recipe_id",
        right_on="id",
        how="left"
    )
    [["recipe_id", "name", "score", "supporting_users"]]
)

top_10_recommendations

In [ ]:
def recommend_for_user(user_id, k=10, n_similar_users=10):
    # Proveri da li korisnik postoji u trening skupu
    if user_id not in user_to_index:
        return pd.DataFrame(columns=["recipe_id", "name", "score", "supporting_users"])

    # Indeks korisnika u user-item matrici
    user_index = user_to_index[user_id]

    # Sličnost sa svim korisnicima
    user_similarities = cosine_similarity(
        user_item_matrix[user_index],
        user_item_matrix
    ).flatten()

    # Izbaci samog korisnika
    user_similarities[user_index] = -1

    # Indeksi najsličnijih korisnika
    similar_indices = np.argsort(user_similarities)[-n_similar_users:][::-1]

    # Tabela sličnih korisnika
    similar_users_df = pd.DataFrame({
        "user_id": [user_ids[i] for i in similar_indices],
        "similarity": [user_similarities[i] for i in similar_indices]
    })

    # Recepti koje je korisnik već ocenio
    seen_recipes = set(
        train_ratings[
            train_ratings["user_id"] == user_id
            ]["recipe_id"]
    )

    # Interakcije sličnih korisnika
    candidates = train_ratings[
        train_ratings["user_id"].isin(similar_users_df["user_id"])
    ].copy()

    # Izbaci već viđene recepte
    candidates = candidates[
        ~candidates["recipe_id"].isin(seen_recipes)
    ].copy()

    # Dodaj sličnost korisnika
    candidates = candidates.merge(
        similar_users_df,
        on="user_id",
        how="left"
    )

    # Weighted score
    candidates["weighted_rating"] = (
            candidates["rating"] * candidates["similarity"]
    )

    # Rangiranje recepata
    scores = (
        candidates
        .groupby("recipe_id")
        .agg(
            score=("weighted_rating", "sum"),
            supporting_users=("user_id", "nunique")
        )
        .reset_index()
        .sort_values(
            ["score", "supporting_users"],
            ascending=[False, False]
        )
        .head(k)
    )

    # Dodaj naziv recepta
    recommendations = scores.merge(
        recipes[["id", "name"]],
        left_on="recipe_id",
        right_on="id",
        how="left"
    )

    return recommendations[
        ["recipe_id", "name", "score", "supporting_users"]
    ]

In [ ]:
recommendations = recommend_for_user(test_user_id, k=10)

recommendations

In [ ]:
def evaluate_collaborative_filtering(k=10, max_users=1000):
    precisions = []

    # Korisnici koji postoje u test skupu
    test_users = test_ratings["user_id"].unique()

    # Ograničavamo broj korisnika zbog vremena izvršavanja
    test_users = test_users[:max_users]

    for user_id in test_users:
        recommendations = recommend_for_user(
            user_id,
            k=k,
            n_similar_users=10
        )

        if recommendations.empty:
            continue

        recommended_recipes = set(
            recommendations["recipe_id"]
        )

        actual_recipes = set(
            test_ratings[
                test_ratings["user_id"] == user_id
                ]["recipe_id"]
        )

        hits = len(recommended_recipes & actual_recipes)

        precisions.append(hits / k)

    if len(precisions) == 0:
        return 0.0

    return np.mean(precisions)

In [ ]:
cf_precision_at_10 = evaluate_collaborative_filtering(
    k=10,
    max_users=1000
)

print("Collaborative Filtering Precision@10:", cf_precision_at_10)

In [ ]:
def evaluate_collaborative_filtering_recall(k=10, max_users=1000):
    recalls = []

    test_users = test_ratings["user_id"].unique()
    test_users = test_users[:max_users]

    for user_id in test_users:
        recommendations = recommend_for_user(
            user_id,
            k=k,
            n_similar_users=10
        )

        if recommendations.empty:
            continue

        recommended_recipes = set(
            recommendations["recipe_id"]
        )

        actual_recipes = set(
            test_ratings[
                test_ratings["user_id"] == user_id
                ]["recipe_id"]
        )

        if len(actual_recipes) == 0:
            continue

        hits = len(recommended_recipes & actual_recipes)

        recalls.append(hits / len(actual_recipes))

    if len(recalls) == 0:
        return 0.0

    return np.mean(recalls)

In [ ]:
cf_recall_at_10 = evaluate_collaborative_filtering_recall(
    k=10,
    max_users=1000
)

print("Collaborative Filtering Recall@10:", cf_recall_at_10)

In [ ]:
import pickle
import os

cf_model = {
    "user_item_matrix": user_item_matrix,
    "user_ids": user_ids,
    "user_to_index": user_to_index,
    "recipe_ids": recipe_ids,
    "n_similar_users": 10,
    "k": 10,
    "precision_at_10": cf_precision_at_10,
    "recall_at_10": cf_recall_at_10
}

model_path = "../saved_models/collaborative_filtering.pkl"

with open(model_path, "wb") as f:
    pickle.dump(cf_model, f)

print("Collaborative Filtering model saved to:")
print(os.path.abspath(model_path))